# El error más común: atributos mutables de clase

En el vídeo anterior aprendimos la diferencia entre **atributos de instancia**
(propios de cada objeto, definidos sobre `self` en `__init__`) y **atributos de
clase** (compartidos por todas las instancias, definidos en el cuerpo de la clase).

Los atributos de clase esconden una **trampa
peligrosa** cuando el valor compartido es *mutable* (una lista, un diccionario, un
conjunto...). Es una de las fuentes de bugs más desconcertantes al empezar con POO,
porque el código *parece* correcto pero se comporta de forma rarísima.

En este vídeo veremos:

- Por qué aparecen estos "bugs fantasma" en los que varios objetos comparten datos.
- Cuál es la **causa real** del problema.
- La **solución correcta** y cuándo sí tiene sentido un atributo de clase.

---

## 1. Concepto previo: variables mutables e inmutables

Antes de ver la trampa necesitamos un concepto base de Python. Los objetos se dividen
en dos grandes grupos según si se pueden modificar o no una vez creados:

- **Inmutables**: no se pueden cambiar "por dentro". Si parece que los modificas, en
  realidad estás creando un objeto **nuevo**. Por ejemplo: `int`, `float`, `str`,
  `bool`, `tuple`.
- **Mutables**: se pueden modificar en el sitio, sin crear uno nuevo. Por ejemplo:
  `list`, `dict`, `set`.

Podemos verlo con `id()`, que nos da una especie de "identidad" del objeto en memoria:
si el `id` cambia, es que estamos ante un objeto distinto.

In [ ]:
# INMUTABLE: al "modificar" un string obtenemos en realidad un objeto NUEVO
texto = "hola"
print("Antes:  ", id(texto))
texto += " mundo"
print("Después:", id(texto))  # id DISTINTO -> es otro objeto

print()

# MUTABLE: la lista se modifica EN EL SITIO; sigue siendo el mismo objeto
numeros = [1, 2, 3]
print("Antes:  ", id(numeros))
numeros.append(4)
print("Después:", id(numeros))  # MISMO id -> la misma lista, cambiada por dentro

El `id` del *string* cambia (es otro objeto), pero el de la lista se mantiene. Esta
distinción es la **clave** de todo el vídeo: un atributo de clase **inmutable** (un
número, un texto) rara vez da problemas, pero uno **mutable** (una lista, un
diccionario) compartido entre instancias es justo donde aparece el bug que vamos a ver
a continuación.

---

## 2. El bug en vivo

Queremos que cada equipo tenga su propia lista de miembros. Una forma **aparentemente
razonable** de escribirlo sería esta:

In [ ]:
class Equipo:
    miembros = []  # ⚠️ ¡cuidado! lista como atributo de CLASE

    def __init__(self, nombre):
        self.nombre = nombre

    def añadir(self, persona):
        self.miembros.append(persona)


leones = Equipo("Leones")
tigres = Equipo("Tigres")

# Solo añadimos a Ana al equipo de los Leones...
leones.añadir("Ana")

print("Leones:", leones.miembros)
print("Tigres:", tigres.miembros)

¡Sorpresa! Ana aparece **también** en los Tigres, aunque solo la añadimos a los
Leones. Y si creamos un tercer equipo, también la tendrá:

In [ ]:
osos = Equipo("Osos")
print("Osos (recién creado):", osos.miembros)  # ['Ana'] ¡¿cómo?!

---

## 3. ¿Por qué ocurre esto?

La razón conecta directamente con lo que acabamos de ver: el atributo `miembros` se
define en el **cuerpo de la clase**, así que **existe una sola lista** y todas las
instancias apuntan a ella.

Cuando hacemos `self.miembros.append(...)`, no estamos creando nada nuevo: estamos
**modificando esa única lista compartida** (recuerda, las listas son **mutables**).
Podemos comprobarlo viendo que es literalmente el mismo objeto en memoria:

In [ ]:
# Es EL MISMO objeto lista para todos (mismo id en memoria)
print(leones.miembros is tigres.miembros)  # True
print(id(leones.miembros) == id(tigres.miembros))  # True

# Y es la propia lista de la clase
print(leones.miembros is Equipo.miembros)  # True

> 💡 Aquí se ve la conexión con la primera sección: si en vez de una lista el
> atributo fuera **inmutable** (un texto, un número), no habría problema. Al hacer
> `equipo.deporte = "baloncesto"` no cambiaríamos ninguna cadena compartida, sino que
> crearíamos un **atributo de instancia nuevo** en ese objeto que tapa al de la clase.
> La lista, en cambio, la mutamos en el sitio (`.append`) y el cambio lo ven todos.
> Por eso la trampa **solo se manifiesta con tipos mutables**.

---

## 4. La solución correcta: estado mutable en `__init__`

La regla es sencilla: **el estado mutable propio de cada objeto se inicializa dentro
de `__init__`**. Así, cada instancia crea su **propia** lista nueva:

In [ ]:
class Equipo:
    def __init__(self, nombre):
        self.nombre = nombre
        self.miembros = []  # ✅ una lista NUEVA para cada equipo

    def añadir(self, persona):
        self.miembros.append(persona)


leones = Equipo("Leones")
tigres = Equipo("Tigres")

leones.añadir("Ana")

print("Leones:", leones.miembros)  # ['Ana']
print("Tigres:", tigres.miembros)  # []  -> ¡correcto!

In [ ]:
# Ahora cada equipo tiene su propia lista (objetos distintos)
print(leones.miembros is tigres.miembros)  # False

---

## 5. Resumen

En este vídeo hemos desmontado uno de los errores más comunes al empezar con POO:

1. Un atributo de clase **mutable** (una lista, un diccionario...) se **comparte**
   entre todas las instancias: una sola copia para todas.
2. Por eso, al modificarlo desde un objeto (`self.miembros.append(...)`), el cambio
   lo "ven" todos los demás: el clásico **bug de estado compartido**.
3. La solución es inicializar el **estado mutable propio de cada objeto dentro de
   `__init__`** (sobre `self`), para que cada instancia tenga el suyo.
4. Los atributos de clase siguen siendo útiles para **constantes** y **datos
   compartidos a propósito** (como un contador).
5. Y no todo tiene por qué ser una clase: para datos simples, un **diccionario**
   puede ser la mejor opción.

| Quieres que el dato sea... | Dónde lo defines |
|----------------------------|------------------|
| Propio de cada objeto (sobre todo si es mutable) | En `__init__`, sobre `self` |
| Compartido por toda la clase (constante / contador) | En el cuerpo de la clase |